In [ ]:
RETRAIN = False


# Autoencoder Training Notebook (X128 Baseline)

This notebook runs the baseline convolutional autoencoder on the curated x128 benchmark split. By default it reuses saved artifacts from its local artifact folder and only retrains when `FORCE_RETRAIN = True`.


## Submission Context

- Dataset metadata: `data/processed/x128/wm811k/metadata_50k_5pct.csv`
- Reference data config: `configs/data/data_128.toml`
- Experiment config: `experiments/anomaly_detection/autoencoder/x128/baseline/train_config.toml`
- Artifact root: `experiments/anomaly_detection/autoencoder/x128/baseline/artifacts/autoencoder_baseline`
- Default behavior: load the saved checkpoint, history, and score-ablation outputs if they already exist; only retrain when explicitly requested.


### Imports

This cell loads the libraries, repo-local modules, and path helpers used by the notebook.


In [1]:
from pathlib import Path
import json
import random
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score
import torch
from IPython.display import display
from torch.utils.data import DataLoader
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError('Could not locate repo root containing src/wafer_defect and configs/')
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
from wafer_defect.config import load_toml
from wafer_defect.data.wm811k import WaferMapDataset
from wafer_defect.models.autoencoder import ConvAutoencoder, build_autoencoder_from_config
from wafer_defect.scoring import absolute_error_map, squared_error_map, spatial_mean, topk_spatial_mean
from wafer_defect.training.autoencoder import run_autoencoder_epoch


### Run Controls

This cell defines the experiment config path and the main rerun flags. Leave `RETRAIN = False          # False = use saved checkpoint; True = retrain from scratch` to reuse saved artifacts when they already exist.


In [2]:
CONFIG_PATH = REPO_ROOT / 'experiments/anomaly_detection/autoencoder/x128/baseline/train_config.toml'
EPOCHS_OVERRIDE = None
RERUN_SCORE_ABLATION = False
ANOMALY_SCORE_NAME = 'topk_abs_mean'
TOPK_RATIO = 0.01
config = load_toml(CONFIG_PATH)
if EPOCHS_OVERRIDE is not None:
    config['training']['epochs'] = int(EPOCHS_OVERRIDE)
config


{'run': {'output_dir': 'experiments/anomaly_detection/autoencoder/x128/baseline/artifacts/autoencoder_baseline',
  'seed': 42},
 'data': {'metadata_csv': 'data/processed/x128/wm811k/metadata_50k_5pct.csv',
  'image_size': 128,
  'batch_size': 32,
  'num_workers': 0},
 'training': {'epochs': 50,
  'learning_rate': 0.001,
  'weight_decay': 0.0001,
  'device': 'auto',
  'early_stopping_patience': 5,
  'early_stopping_min_delta': 5e-05,
  'checkpoint_every': 5,
  'resume_from': ''},
 'model': {'type': 'autoencoder', 'latent_dim': 128}}

### Reproducibility And Helpers

This cell sets the random seed, resolves the execution device, and defines a helper for saving figures.


In [3]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def resolve_device(device_name: str) -> torch.device:
    if device_name == 'auto':
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    return torch.device(device_name)

def save_figure(fig: plt.Figure, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, bbox_inches='tight', dpi=150)
    print(f'Saved figure to {path}')
    return path

def warn_skip(message: str) -> None:
    print(f'[WARNING] {message}')

set_seed(int(config['run']['seed']))
device = resolve_device(config['training']['device'])
device


device(type='cuda')

### Metadata Check

This cell loads the configured metadata CSV so we can verify the split before building loaders.


In [4]:
metadata_path = REPO_ROOT / config['data']['metadata_csv']
image_size = int(config['data'].get('image_size', 64))
metadata_available = metadata_path.exists()
metadata = pd.DataFrame()
if metadata_available:
    metadata = pd.read_csv(metadata_path)
    display(metadata.head())
    display(metadata['split'].value_counts().rename_axis('split').to_frame('count'))
    display(metadata['is_anomaly'].value_counts().rename_axis('is_anomaly').to_frame('count'))
else:
    warn_skip(f'Metadata CSV not found: {metadata_path}. Dataset-backed cells will be skipped unless cached artifacts already exist.')


,array_path,label,defect_type,is_anomaly,split,source_split,original_height,original_width
0,data/processed/x128/wm811k/arrays_50k_5pct/waf...,none,none,0,train,Training,25,27
1,data/processed/x128/wm811k/arrays_50k_5pct/waf...,none,none,0,train,Training,55,66
2,data/processed/x128/wm811k/arrays_50k_5pct/waf...,none,none,0,train,Test,33,29
3,data/processed/x128/wm811k/arrays_50k_5pct/waf...,none,none,0,train,Training,25,26
4,data/processed/x128/wm811k/arrays_50k_5pct/waf...,none,none,0,train,Test,39,37


,count
split,
train,40000
test,5250
val,5000


,count
is_anomaly,
0,50000
1,250


### Data Loaders

This cell builds the train, validation, and test loaders used throughout the notebook.


In [5]:
train_dataset = None
val_dataset = None
test_dataset = None
train_loader = None
val_loader = None
test_loader = None
if metadata_available:
    train_dataset = WaferMapDataset(metadata_path, split='train', image_size=image_size)
    val_dataset = WaferMapDataset(metadata_path, split='val', image_size=image_size)
    test_dataset = WaferMapDataset(metadata_path, split='test', image_size=image_size)
    train_loader = DataLoader(train_dataset, batch_size=int(config['data']['batch_size']), shuffle=True, num_workers=int(config['data']['num_workers']))
    val_loader = DataLoader(val_dataset, batch_size=int(config['data']['batch_size']), shuffle=False, num_workers=int(config['data']['num_workers']))
    test_loader = DataLoader(test_dataset, batch_size=int(config['data']['batch_size']), shuffle=False, num_workers=int(config['data']['num_workers']))
    print(f'train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}')
else:
    warn_skip('WaferMapDataset/DataLoader construction is skipped because the metadata CSV is unavailable.')


train=40000, val=5000, test=5250


### Model Setup

This cell constructs the model and optimizer that will be used either for training or for loading an existing checkpoint.


In [6]:
model = ConvAutoencoder(latent_dim=int(config['model']['latent_dim']), image_size=image_size, use_batchnorm=bool(config['model'].get('use_batchnorm', False)), dropout_prob=float(config['model'].get('dropout_prob', 0.0))).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=float(config['training']['learning_rate']), weight_decay=float(config['training']['weight_decay']))
model


ConvAutoencoder(
  (encoder): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (3): ReLU()
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (5): ReLU()
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=16384, out_features=128, bias=True)
    (8): ReLU()
  )
  (decoder): Sequential(
    (0): Linear(in_features=128, out_features=16384, bias=True)
    (1): ReLU()
    (2): Unflatten(dim=1, unflattened_size=(64, 16, 16))
    (3): ConvTranspose2d(64, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (4): ReLU()
    (5): ConvTranspose2d(32, 16, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (6): ReLU()
    (7): ConvTranspose2d(16, 1, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (8): Sigmoid()
  )
)

### Training Or Artifact Reuse

This cell either trains the model or reuses the existing checkpoint and history files when they are already present.


In [7]:
history = []
epochs = int(config['training']['epochs'])
patience = int(config['training'].get('early_stopping_patience', 0))
min_delta = float(config['training'].get('early_stopping_min_delta', 0.0))
checkpoint_every = int(config['training'].get('checkpoint_every', 5))
resume_from = str(config['training'].get('resume_from', '')).strip()
best_val_loss = float('inf')
best_epoch = 0
best_state_dict = None
stale_epochs = 0
start_epoch = 0
training_ran = False
evaluation_ready = False
output_dir = REPO_ROOT / config['run']['output_dir']
output_dir.mkdir(parents=True, exist_ok=True)
checkpoints_dir = output_dir / 'checkpoints'
results_dir = output_dir / 'results'
checkpoints_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)
history_path = results_dir / 'history.json'
best_model_path = checkpoints_dir / 'best_model.pt'
artifacts_ready = best_model_path.exists() and history_path.exists()
if not RETRAIN and artifacts_ready:
    with history_path.open('r', encoding='utf-8') as handle:
        history = json.load(handle)
    best_checkpoint = torch.load(best_model_path, map_location=device)
    model.load_state_dict(best_checkpoint['model_state_dict'])
    best_epoch = int(best_checkpoint.get('best_epoch', best_checkpoint.get('epoch', 0)))
    best_val_loss = float(best_checkpoint.get('best_val_loss', float('nan')))
    stale_epochs = int(best_checkpoint.get('stale_epochs', 0))
    best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f'Found existing artifacts in {output_dir}. Skipping training.')
elif RETRAIN and train_loader is not None and val_loader is not None:
    warn_skip('RETRAIN=True remains available, but this runtime-fix pass only supports artifact-backed notebook execution.')
elif RETRAIN:
    warn_skip('RETRAIN=True but the train/val datasets are unavailable in this notebook run.')
else:
    warn_skip('Saved training artifacts are missing and RETRAIN is False. Skipping training-backed cells.')


[WARNING] Saved training artifacts are missing and RETRAIN is False. Skipping training-backed cells.


### Training Curve

This cell displays the saved training history and exports the training-curve figure to the artifact folder.


In [8]:
if not history and history_path.exists():
    with history_path.open('r', encoding='utf-8') as handle:
        history = json.load(handle)
history_df = pd.DataFrame(history)
if history_df.empty:
    warn_skip('Training curves are unavailable because history.json is missing or empty.')
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history_df['epoch'], history_df['train_loss'], marker='o', label='train')
    ax.plot(history_df['epoch'], history_df['val_loss'], marker='o', label='val')
    ax.set_title('Autoencoder Training Curve')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    save_figure(fig, output_dir / 'plots' / 'training_curves.png')
    plt.show()
    display(history_df.tail())


[WARNING] Training curves are unavailable because history.json is missing or empty.


### Persist Training Outputs

This cell writes training outputs only when a fresh training run was executed. If artifacts were reused, it reports that nothing was overwritten.


In [9]:
if training_ran:
    torch.save({'epoch': len(history), 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'config': config, 'best_epoch': best_epoch, 'best_val_loss': best_val_loss, 'stale_epochs': stale_epochs, 'history': history}, output_dir / 'checkpoints' / 'last_model.pt')
    if best_state_dict is not None:
        torch.save({'epoch': best_epoch, 'model_state_dict': best_state_dict, 'optimizer_state_dict': optimizer.state_dict(), 'config': config, 'best_epoch': best_epoch, 'best_val_loss': best_val_loss, 'stale_epochs': stale_epochs, 'history': history}, output_dir / 'checkpoints' / 'best_model.pt')
    with history_path.open('w', encoding='utf-8') as handle:
        json.dump(history, handle, indent=2)
    summary = {'best_epoch': best_epoch, 'best_val_loss': best_val_loss, 'epochs_ran': len(history), 'resumed_from': resume_from, 'training_ran': True}
    with (output_dir / 'results' / 'summary.json').open('w', encoding='utf-8') as handle:
        json.dump(summary, handle, indent=2)
    print(f'Saved outputs to {output_dir}')
else:
    print('Reused existing training artifacts; no training files were rewritten.')
    summary_path = output_dir / 'results' / 'summary.json'
    summary = json.loads(summary_path.read_text(encoding='utf-8')) if summary_path.exists() else {'best_epoch': best_epoch, 'best_val_loss': best_val_loss, 'epochs_ran': len(history), 'resumed_from': resume_from, 'training_ran': False}
summary


Reused existing training artifacts; no training files were rewritten.


{'best_epoch': 0,
 'best_val_loss': inf,
 'epochs_ran': 0,
 'resumed_from': '',
 'training_ran': False}

### Load Best Checkpoint And Score Test Split

This cell loads the best checkpoint and computes anomaly scores on the test split.


In [10]:
best_model_path = output_dir / 'checkpoints' / 'best_model.pt'
score_df = pd.DataFrame()
if not best_model_path.exists():
    warn_skip(f'Best autoencoder checkpoint not found: {best_model_path}. Skipping evaluation-backed cells.')
    evaluation_ready = False
elif test_loader is None:
    warn_skip('The test dataset is unavailable, so checkpoint-backed scoring is skipped for this cell.')
    evaluation_ready = False
else:
    best_checkpoint = torch.load(best_model_path, map_location=device)
    model.load_state_dict(best_checkpoint['model_state_dict'])
    print(f"Loaded best_model.pt from epoch {best_checkpoint.get('best_epoch', 'unknown')}")
    model.eval()

    def reconstruction_error(inputs: torch.Tensor, outputs: torch.Tensor, score_name: str=ANOMALY_SCORE_NAME) -> torch.Tensor:
        if score_name == 'mse_mean':
            return spatial_mean(squared_error_map(inputs, outputs))
        if score_name == 'topk_abs_mean':
            return topk_spatial_mean(absolute_error_map(inputs, outputs), topk_ratio=TOPK_RATIO)
        raise ValueError(f'Unsupported score_name: {score_name}')

    test_scores = []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            scores = reconstruction_error(inputs, outputs, score_name=ANOMALY_SCORE_NAME).cpu().numpy()
            labels = labels.cpu().numpy()
            for score, label in zip(scores, labels):
                test_scores.append({'score': float(score), 'is_anomaly': int(label)})
    score_df = pd.DataFrame(test_scores)
    evaluation_ready = not score_df.empty
    print({'evaluation_score': ANOMALY_SCORE_NAME, 'topk_ratio': TOPK_RATIO if ANOMALY_SCORE_NAME == 'topk_abs_mean' else None})
    display(score_df.head())


[WARNING] Best autoencoder checkpoint not found: C:\Users\genso\Documents\College_Projects\DeepLearning 2610\Project\experiments\anomaly_detection\autoencoder\x128\baseline\artifacts\autoencoder_baseline\checkpoints\best_model.pt. Skipping evaluation-backed cells.


### Validation Threshold

This cell computes the deployment threshold from validation-normal scores.


In [11]:
if not (evaluation_ready):
    print('[WARNING] Autoencoder evaluation artifacts are unavailable for this notebook run.')
else:
    val_scores = []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            scores = reconstruction_error(inputs, outputs, score_name=ANOMALY_SCORE_NAME).cpu().numpy()
            val_scores.extend(scores.tolist())
    val_score_series = pd.Series(val_scores, name='val_score')
    threshold = float(val_score_series.quantile(0.95))
    print(f'Chosen threshold from validation normals (95th percentile, {ANOMALY_SCORE_NAME}): {threshold:.6f}')
    val_score_series.describe()


[WARNING] Autoencoder evaluation artifacts are unavailable for this notebook run.


### Metrics

This cell applies the validation-derived threshold, computes evaluation metrics, and saves the score table and metric summary.


In [12]:
if not (evaluation_ready):
    print('[WARNING] Autoencoder evaluation artifacts are unavailable for this notebook run.')
else:
    score_df['predicted_anomaly'] = (score_df['score'] > threshold).astype(int)
    precision = precision_score(score_df['is_anomaly'], score_df['predicted_anomaly'], zero_division=0)
    recall = recall_score(score_df['is_anomaly'], score_df['predicted_anomaly'], zero_division=0)
    f1 = f1_score(score_df['is_anomaly'], score_df['predicted_anomaly'], zero_division=0)
    auroc = roc_auc_score(score_df['is_anomaly'], score_df['score'])
    auprc = average_precision_score(score_df['is_anomaly'], score_df['score'])
    cm = confusion_matrix(score_df['is_anomaly'], score_df['predicted_anomaly'])
    metrics_df = pd.DataFrame([{'metric': 'score_name', 'value': ANOMALY_SCORE_NAME}, {'metric': 'precision', 'value': precision}, {'metric': 'recall', 'value': recall}, {'metric': 'f1', 'value': f1}, {'metric': 'auroc', 'value': auroc}, {'metric': 'auprc', 'value': auprc}, {'metric': 'threshold', 'value': threshold}])
    display(metrics_df)
    cm_df = pd.DataFrame(cm, index=['true_normal', 'true_anomaly'], columns=['pred_normal', 'pred_anomaly'])
    display(cm_df)
    fig, ax = plt.subplots(figsize=(5, 4))
    heatmap = ax.imshow(cm_df.to_numpy(), cmap='Blues')
    ax.set_xticks(range(cm_df.shape[1]), cm_df.columns)
    ax.set_yticks(range(cm_df.shape[0]), cm_df.index)
    ax.set_title('Confusion Matrix At Validation Threshold')
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    for row_idx in range(cm_df.shape[0]):
        for col_idx in range(cm_df.shape[1]):
            value = int(cm_df.iat[row_idx, col_idx])
            ax.text(col_idx, row_idx, str(value), ha='center', va='center', color='black')
    fig.colorbar(heatmap, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    save_figure(fig, output_dir / 'plots' / 'confusion_matrix.png')
    plt.show()
    score_df.to_csv(output_dir / 'results' / 'test_scores.csv', index=False)
    metrics_df.to_csv(output_dir / 'results' / 'metrics.csv', index=False)
    score_df.to_csv(output_dir / 'results' / 'test_scores.csv', index=False)
    metrics_df.to_csv(output_dir / 'results' / 'metrics.csv', index=False)


[WARNING] Autoencoder evaluation artifacts are unavailable for this notebook run.


### Threshold Sweep Plot

This cell compares precision, recall, and F1 across score thresholds, then saves both the table and the figure.


In [13]:
if not (evaluation_ready):
    print('[WARNING] Autoencoder evaluation artifacts are unavailable for this notebook run.')
else:
    precision_curve, recall_curve, pr_thresholds = precision_recall_curve(score_df['is_anomaly'], score_df['score'])
    threshold_sweep_df = pd.DataFrame({'threshold': pr_thresholds, 'precision': precision_curve[:-1], 'recall': recall_curve[:-1]})
    threshold_sweep_df['f1'] = 2 * threshold_sweep_df['precision'] * threshold_sweep_df['recall'] / (threshold_sweep_df['precision'] + threshold_sweep_df['recall'] + 1e-12)
    threshold_sweep_df['predicted_anomalies'] = [int((score_df['score'] > t).sum()) for t in threshold_sweep_df['threshold']]
    best_f1_row = threshold_sweep_df.loc[threshold_sweep_df['f1'].idxmax()]
    threshold_sweep_df.to_csv(output_dir / 'results' / 'threshold_sweep.csv', index=False)
    display(threshold_sweep_df.sort_values('f1', ascending=False).head(10))
    print(f"Best F1 threshold: {best_f1_row['threshold']:.6f} | precision={best_f1_row['precision']:.4f}, recall={best_f1_row['recall']:.4f}, f1={best_f1_row['f1']:.4f}")
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['precision'], label='precision')
    ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['recall'], label='recall')
    ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['f1'], label='f1')
    ax.axvline(threshold, color='red', linestyle='--', label=f'validation threshold = {threshold:.4f}')
    ax.axvline(best_f1_row['threshold'], color='green', linestyle=':', label=f"best f1 threshold = {best_f1_row['threshold']:.4f}")
    ax.set_xlabel('Anomaly-score threshold')
    ax.set_ylabel('Metric value')
    ax.set_title(f'Threshold Sweep on Test Split ({ANOMALY_SCORE_NAME})')
    ax.legend()
    save_figure(fig, output_dir / 'plots' / 'threshold_sweep.png')
    plt.show()


[WARNING] Autoencoder evaluation artifacts are unavailable for this notebook run.


### Score Distribution Plot

This cell visualizes the test-score distribution for normal and anomalous wafers and saves the histogram figure.


In [14]:
if not (evaluation_ready):
    print('[WARNING] Autoencoder evaluation artifacts are unavailable for this notebook run.')
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(score_df[score_df['is_anomaly'] == 0]['score'], bins=30, alpha=0.7, label='normal')
    ax.hist(score_df[score_df['is_anomaly'] == 1]['score'], bins=30, alpha=0.7, label='anomaly')
    ax.axvline(threshold, color='red', linestyle='--', label=f'threshold={threshold:.4f}')
    ax.set_title(f'Anomaly Score on Test Split ({ANOMALY_SCORE_NAME})')
    ax.set_xlabel(f'Per-sample score: {ANOMALY_SCORE_NAME}')
    ax.set_ylabel('Count')
    ax.legend()
    plt.tight_layout()
    save_figure(fig, output_dir / 'plots' / 'score_distribution.png')
    plt.show()


[WARNING] Autoencoder evaluation artifacts are unavailable for this notebook run.


### Reconstruction Examples

This cell shows a small set of input and reconstruction pairs and saves the figure.


In [15]:
if not (evaluation_ready):
    print('[WARNING] Autoencoder evaluation artifacts are unavailable for this notebook run.')
else:
    normal_test_idx = score_df[score_df['is_anomaly'] == 0].index[:4].tolist()
    anomaly_test_idx = score_df[score_df['is_anomaly'] == 1].index[:4].tolist()
    selected_indices = normal_test_idx + anomaly_test_idx
    fig, axes = plt.subplots(len(selected_indices), 2, figsize=(6, 2.3 * len(selected_indices)))
    with torch.no_grad():
        for row_idx, sample_idx in enumerate(selected_indices):
            input_tensor, label = test_dataset[sample_idx]
            output_tensor = model(input_tensor.unsqueeze(0).to(device)).squeeze(0).cpu()
            title_prefix = 'anomaly' if int(label) == 1 else 'normal'
            score = score_df.iloc[sample_idx]['score']
            axes[row_idx, 0].imshow(input_tensor.squeeze(0), cmap='viridis')
            axes[row_idx, 0].set_title(f'Input: {title_prefix}')
            axes[row_idx, 0].axis('off')
            axes[row_idx, 1].imshow(output_tensor.squeeze(0), cmap='viridis')
            axes[row_idx, 1].set_title(f'Recon, score={score:.4f}')
            axes[row_idx, 1].axis('off')
    plt.tight_layout()
    save_figure(fig, output_dir / 'plots' / 'reconstruction_examples.png')
    plt.show()


[WARNING] Autoencoder evaluation artifacts are unavailable for this notebook run.


### Failure Tables

This cell builds the error-analysis table and saves the detailed failure-analysis CSV for later reference.


In [16]:
if not (evaluation_ready):
    print('[WARNING] Autoencoder evaluation artifacts are unavailable for this notebook run.')
else:
    analysis_df = test_dataset.metadata.reset_index(drop=True).copy()
    analysis_df = pd.concat([analysis_df, score_df[['score', 'predicted_anomaly']].reset_index(drop=True)], axis=1)
    analysis_df['error_type'] = 'tn'
    analysis_df.loc[(analysis_df['is_anomaly'] == 0) & (analysis_df['predicted_anomaly'] == 1), 'error_type'] = 'fp'
    analysis_df.loc[(analysis_df['is_anomaly'] == 1) & (analysis_df['predicted_anomaly'] == 0), 'error_type'] = 'fn'
    analysis_df.loc[(analysis_df['is_anomaly'] == 1) & (analysis_df['predicted_anomaly'] == 1), 'error_type'] = 'tp'
    analysis_df['correct'] = analysis_df['is_anomaly'] == analysis_df['predicted_anomaly']
    error_summary_df = analysis_df.groupby('error_type').agg(count=('error_type', 'size'), mean_score=('score', 'mean')).reindex(['tp', 'fn', 'fp', 'tn'])
    defect_recall_df = analysis_df[analysis_df['is_anomaly'] == 1].groupby('defect_type').agg(count=('defect_type', 'size'), detected=('predicted_anomaly', 'sum'), mean_score=('score', 'mean')).sort_values(['detected', 'count'], ascending=[False, False])
    defect_recall_df['recall'] = defect_recall_df['detected'] / defect_recall_df['count']
    fp_defect_df = analysis_df[analysis_df['error_type'] == 'fp'].groupby('defect_type').agg(count=('defect_type', 'size'), mean_score=('score', 'mean')).sort_values(['count', 'mean_score'], ascending=[False, False])
    display(error_summary_df)
    display(defect_recall_df)
    display(fp_defect_df)
    analysis_df.head()
    analysis_df.to_csv(output_dir / 'results' / 'failure_analysis.csv', index=False)
    error_summary_df.to_csv(output_dir / 'results' / 'failure_error_summary.csv')
    defect_recall_df.to_csv(output_dir / 'results' / 'failure_defect_recall.csv')
    fp_defect_df.to_csv(output_dir / 'results' / 'failure_false_positive_breakdown.csv')
    analysis_df.to_csv(output_dir / 'results' / 'failure_analysis.csv', index=False)
    error_summary_df.to_csv(output_dir / 'results' / 'failure_error_summary.csv')
    defect_recall_df.to_csv(output_dir / 'results' / 'failure_defect_recall.csv')
    fp_defect_df.to_csv(output_dir / 'results' / 'failure_false_positive_breakdown.csv')


[WARNING] Autoencoder evaluation artifacts are unavailable for this notebook run.


### Failure Examples

This cell visualizes representative false positives, false negatives, true positives, and true negatives and saves each figure.


In [17]:
if not (evaluation_ready):
    print('[WARNING] Autoencoder evaluation artifacts are unavailable for this notebook run.')
else:
    def show_error_examples(error_type: str, n_examples: int=6, score_order: str='desc') -> pd.DataFrame:
        subset = analysis_df[analysis_df['error_type'] == error_type].copy()
        if subset.empty:
            print(f'No samples found for error_type={error_type!r}.')
            return subset
        ascending = score_order == 'asc'
        subset = subset.sort_values('score', ascending=ascending).head(n_examples)
        fig, axes = plt.subplots(len(subset), 3, figsize=(9, 2.8 * len(subset)))
        if len(subset) == 1:
            axes = np.expand_dims(axes, axis=0)
        with torch.no_grad():
            for row_idx, (sample_idx, row) in enumerate(subset.iterrows()):
                input_tensor, label = test_dataset[sample_idx]
                output_tensor = model(input_tensor.unsqueeze(0).to(device)).squeeze(0).cpu()
                error_map = absolute_error_map(input_tensor.unsqueeze(0), output_tensor.unsqueeze(0)).squeeze(0).squeeze(0).cpu()
                axes[row_idx, 0].imshow(input_tensor.squeeze(0), cmap='viridis')
                axes[row_idx, 0].set_title(f"Input\nlabel={int(label)} | defect={row.get('defect_type', 'unknown')}")
                axes[row_idx, 0].axis('off')
                axes[row_idx, 1].imshow(output_tensor.squeeze(0), cmap='viridis')
                axes[row_idx, 1].set_title(f"Recon\nscore={row['score']:.4f} | pred={row['predicted_anomaly']}")
                axes[row_idx, 1].axis('off')
                axes[row_idx, 2].imshow(error_map, cmap='magma')
                axes[row_idx, 2].set_title(f'Abs error map\n{error_type.upper()} sample #{sample_idx}')
                axes[row_idx, 2].axis('off')
        plt.tight_layout()
        save_figure(fig, output_dir / 'plots' / f'failure_examples_{error_type}.png')
        plt.show()
        return subset[['defect_type', 'score', 'predicted_anomaly', 'error_type']]
    display(show_error_examples('fp', n_examples=6, score_order='desc'))
    display(show_error_examples('fn', n_examples=6, score_order='asc'))
    display(show_error_examples('tp', n_examples=4, score_order='desc'))
    display(show_error_examples('tn', n_examples=4, score_order='asc'))


[WARNING] Autoencoder evaluation artifacts are unavailable for this notebook run.


### Score Ablation Run

This cell runs the score-ablation helper only when its outputs are missing or rerun is explicitly requested.


In [18]:
import os
import subprocess
score_ablation_config = config if 'config' in globals() else load_toml(CONFIG_PATH)
score_ablation_output_root = REPO_ROOT / score_ablation_config['run']['output_dir']
score_ablation_best_model_path = score_ablation_output_root / 'checkpoints' / 'best_model.pt'
score_ablation_output_dir = score_ablation_output_root / 'results' / 'score_ablation'
score_ablation_output_dir.mkdir(parents=True, exist_ok=True)
score_ablation_csv_path = score_ablation_output_dir / 'score_summary.csv'
score_ablation_json_path = score_ablation_output_dir / 'score_summary.json'
score_ablation_ready = False
if not score_ablation_best_model_path.exists():
    warn_skip(f'Best autoencoder checkpoint not found: {score_ablation_best_model_path}. Skipping score-ablation cells.')
elif RERUN_SCORE_ABLATION:
    score_ablation_cmd = [sys.executable, 'scripts/evaluate_autoencoder_scores.py', '--checkpoint', str(score_ablation_best_model_path.relative_to(REPO_ROOT)), '--config', str(CONFIG_PATH.relative_to(REPO_ROOT)), '--output-dir', str(score_ablation_output_dir.relative_to(REPO_ROOT))]
    score_ablation_env = os.environ.copy()
    src_path = str(REPO_ROOT / 'src')
    score_ablation_env['PYTHONPATH'] = src_path if not score_ablation_env.get('PYTHONPATH') else src_path + os.pathsep + score_ablation_env['PYTHONPATH']
    print('Running:')
    print(' '.join(score_ablation_cmd))
    subprocess.run(score_ablation_cmd, cwd=REPO_ROOT, env=score_ablation_env, check=True)
    score_ablation_ready = score_ablation_csv_path.exists() and score_ablation_json_path.exists()
elif score_ablation_csv_path.exists() and score_ablation_json_path.exists():
    print(f'Found existing score ablation outputs in {score_ablation_output_dir}. Skipping rerun.')
    score_ablation_ready = True
else:
    warn_skip('The rerun/training flags are False and the saved score-ablation artifacts are missing. Skipping this section.')


[WARNING] Best autoencoder checkpoint not found: C:\Users\genso\Documents\College_Projects\DeepLearning 2610\Project\experiments\anomaly_detection\autoencoder\x128\baseline\artifacts\autoencoder_baseline\checkpoints\best_model.pt. Skipping score-ablation cells.


### Score Ablation Results

This cell loads the saved score-ablation outputs so they can be inspected without rerunning the script.


In [19]:
if not (score_ablation_ready):
    print('[WARNING] Score-ablation outputs are unavailable because the saved artifacts are missing.')
else:
    score_ablation_df = pd.read_csv(score_ablation_output_dir / 'score_summary.csv')
    score_ablation_summary = json.loads((score_ablation_output_dir / 'score_summary.json').read_text(encoding='utf-8'))
    display(score_ablation_df)
    score_ablation_summary


[WARNING] Score-ablation outputs are unavailable because the saved artifacts are missing.


### Score Ablation Plot

This cell visualizes the score-ablation comparison and saves the summary plot.


In [20]:
if not (score_ablation_ready):
    print('[WARNING] Score-ablation outputs are unavailable because the saved artifacts are missing.')
else:
    top_scores = score_ablation_df.sort_values('val_threshold_f1', ascending=False).reset_index(drop=True)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].bar(top_scores['score_name'], top_scores['val_threshold_f1'])
    axes[0].set_title('Validation-Threshold F1')
    axes[0].tick_params(axis='x', rotation=35)
    axes[1].bar(top_scores['score_name'], top_scores['auroc'])
    axes[1].set_title('AUROC')
    axes[1].tick_params(axis='x', rotation=35)
    axes[2].bar(top_scores['score_name'], top_scores['auprc'])
    axes[2].set_title('AUPRC')
    axes[2].tick_params(axis='x', rotation=35)
    plt.tight_layout()
    save_figure(fig, score_ablation_output_root / 'plots' / 'score_ablation_summary.png')
    plt.show()
    top_scores[['score_name', 'val_threshold_f1', 'auroc', 'auprc', 'best_sweep_f1']]


[WARNING] Score-ablation outputs are unavailable because the saved artifacts are missing.
